# Heart Studio — Colab render runner (wake procedure)

Mount this Colab GPU as a self-hosted GitHub Actions runner for `proa28518-creator/grayline-utils`, labeled `colab-gpu`.

Wake steps: sign in to Colab as the seat account, open this notebook, select an L4 (or T4) GPU runtime, paste a fresh GitHub runner registration token into the config cell, then Run all.

Registration tokens live about 1 hour; generate a fresh one at wake time (repo → Settings → Actions → Runners → New self-hosted runner, or the GitHub API registration-token endpoint). NEVER commit a token. The session ends at ≤12 h or on idle. Stop via Runtime → Disconnect and delete runtime.

## 1 — Configuration
Paste the fresh token; choose the name.

In [ ]:
REPO = "proa28518-creator/grayline-utils"
RUNNER_TOKEN = ""
LABELS = "colab-gpu"
NAME = "colab-t4"
assert RUNNER_TOKEN, "paste a fresh registration token first"

## 2 — GPU check (Runtime → Change runtime type → L4 preferred, T4 fallback)

In [ ]:
!nvidia-smi

## 3 — Install the Actions runner (latest release, linux-x64)

In [ ]:
import json, os, tarfile, urllib.request

runner_dir = "/content/actions-runner"
os.makedirs(runner_dir, exist_ok=True)
if not os.path.exists(os.path.join(runner_dir, "config.sh")):
    request = urllib.request.Request("https://api.github.com/repos/actions/runner/releases/latest", headers={"Accept": "application/vnd.github+json", "User-Agent": "Heart-Studio-Colab"})
    with urllib.request.urlopen(request) as response:
        release = json.load(response)
    asset = next(item for item in release["assets"] if item["name"].endswith("linux-x64.tar.gz"))
    archive = "/content/runner.tar.gz"
    urllib.request.urlretrieve(asset["browser_download_url"], archive)
    with tarfile.open(archive, "r:gz") as bundle:
        bundle.extractall(runner_dir)

## 4 — Register (skipped when already registered)

In [ ]:
import os, subprocess

runner_dir = "/content/actions-runner"
if not os.path.exists(os.path.join(runner_dir, ".runner")):
    subprocess.run(["./config.sh", "--url", f"https://github.com/{REPO}", "--token", RUNNER_TOKEN, "--labels", LABELS, "--name", NAME, "--unattended", "--replace"], cwd=runner_dir, check=True)
else:
    print("already registered")

## 5 — Run (blocking — the runner is online while this cell runs)

In [ ]:
%cd /content/actions-runner
!./run.sh